In [ ]:
# Import libraries

import os
import glob
import logging
import json
import time
from datetime import datetime

from dotenv import load_dotenv
import requests
import rasterio
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Initiate logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

load_dotenv()

In [ ]:
################
## User input ##
################

# ORDER DATA: Requires correct folder structure, AOI file (geojson), year range, and target month

# This notebook must be placed in:
# PROJECT FOLDER / NOTEBOOK

# AOI: must be placed in PROJECT FOLDER / DATA / INPUT
aoi_filename = "amazonas.geojson"

# Years and months to consider for the order
year_rage = range(2015,2026)
month = 7

#####

# DOWNLOAD DATA: Requires bands and target month

# I use data for deforestation monitoring from the same months over the years.
# The target month is the start of the dry season: max. cloud cover is already acceptable but no wildland fires yet.
target_month = "07"

# Sentinel-2 bands necessary to calculate NDVI
bands = {
    "red": "b04.tiff",
    "nir": "b08.tiff"
}

In [ ]:
# Check directories

# Input AOI
AOI_PATH = f"../data/input/{aoi_filename}"

# Folder to save raw satellite imagery (if folder doesn't exist, it will be created)
raw_data_dir = "../data/output/raw"
os.makedirs(raw_data_dir, exist_ok=True)

# Folder to save raw satellite imagery (if folder doesn't exist, it will be created)
ndvi_data_dir = "../data/output/ndvi"
os.makedirs(ndvi_data_dir, exist_ok=True)
 

In [ ]:
##############################
## BLOCK 1: TO CREATE ORDER ##
##############################

username = os.getenv("UP42_USERNAME")
password = os.getenv("UP42_PASSWORD")

response = requests.post(
    "https://auth.up42.com/realms/public/protocol/openid-connect/token",
    data={
        "username": username,
        "password": password,
        "grant_type": "password",
        "client_id": "up42-api"
    }
)

if response.status_code == 200:
    token = response.json()["access_token"]
    logger.info("New token. Status code: %s", response.status_code)
else:
    logger.error("Token creation failed: %s", response.status_code)

# Headers for GET requests
headers_get = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

# Headers for POST requests
headers_post = {
    "Authorization": f"Bearer {token}",
    "accept": "application/json",
    "content-type": "application/json"
}

In [ ]:
# List all collections
# Better URL source: https://developer.up42.com/reference/listcollectionsv2

url = "https://api.up42.com/v2/collections"

# Screen first page for content and total number of pages
response = requests.get(url,
                        headers=headers_get,
                        params={"page": 1})

data = response.json()
total_pages = data["totalPages"]
combined_collections = data["content"]

# Loop through pages and append further content
for page in range(2, total_pages+1):
    response = requests.get(url, headers=headers_get, params={"page": page})
    data = response.json()
    combined_collections.extend(data["content"])

if response.status_code == 200:
    logger.info("Authentication successful. Status code: %s", response.status_code)
else:
    logger.error("Authentication failed: %s", response.status_code)

names = [c["name"] for c in combined_collections]

In [ ]:
# Extract Sentinel-related collection from all collections

sentinel_info = []

for c in combined_collections:
    if c["name"] == 'sentinel-2':
        sentinel_info = c

sentinel_collection_name = sentinel_info["name"]
sentinel_collection_id = sentinel_info["dataProducts"][0]["name"]
sentinel_product_id = sentinel_info["dataProducts"][0]["id"]

logger.info("Sentinel collection name: %s", sentinel_collection_name)
logger.info("Sentinel collection ID: %s", sentinel_collection_id)
logger.info("Sentinel collection product ID: %s", sentinel_product_id)

In [ ]:
# List of available scenes
# Search the catalog by host name
# Source: https://developer.up42.com/reference/searchbyhost?utm_source=documentation

# Please note: EULA was accepted on the platform
# Further improvement: accept EULA based on: https://developer.up42.com/reference/accepteuladocumentv2?utm_source=documentation

# Define AOI and timerange based on user input
with open(AOI_PATH) as f:
    aoi = json.load(f)

timeranges = [f"{year}-{month:02d}-01T00:00:00Z/{year}-{month:02d}-28T23:59:59Z" for year in year_rage]

# Select scenes from Sentinel-2 collection matching user input (AOI, year range, month)
# To use comparable data for the index calculation, target month is a single value (see user input)
selected_scene_ids = []

url = "https://api.up42.com/catalog/hosts/earthsearch-aws/stac/search"

for timerange in timeranges:
    logger.info("CURRENT YEAR: %s", timerange)

    payload = {
        "datetime": timerange,
        "limit": 250,
        "intersects": aoi["features"][0]["geometry"],
        "collections": [sentinel_collection_name],
        "query": {
            "cloudCoverage": {"lte": 20}
        }
    }

    response = requests.post(url, json=payload, headers=headers_post)

    if response.status_code != 200:
        logger.error("Request failed for %s : %s", timerange, response.text)
        continue

    data = response.json()

    if not data["features"]:
        logger.error("No scenes found for %s", timerange)
        continue

    # Select scene with lowest cloud cover (within AOI, year range, target month)
    best_scene = None
    best_cloud = float("inf")

    for f in data["features"]:
        props = f["properties"]
        cloud = props["cloudCoverage"]

        if cloud < best_cloud:
                    best_cloud = cloud
                    best_scene = props["sceneId"]

    logger.info("In %s: the scene with lowest cloud cover is: %s with a cloud cover of: %s", timerange, best_scene, best_cloud)

    if best_scene:
        selected_scene_ids.append(best_scene)

logger.info("-" * 50)
logger.info("Total filtered scenes: %s", len(selected_scene_ids))
logger.info("Scene ID list: %s", selected_scene_ids)

In [ ]:
# Create an order
# URL: https://developer.up42.com/reference/createv2orderunderworkspace?utm_source=documentation

# Workspace ID:
workspace_id = os.getenv("UP42_WORKSPACE_ID")

if not workspace_id:
    raise RuntimeError("Missing UP42_WORKSPACE_ID")

# Data product ID:
sentinel_product_id

url = f"https://api.up42.com/v2/orders?workspaceId={workspace_id}"

order_ids = []

for scene_id in selected_scene_ids:
    payload = {
        "displayName": f"Sentinel-2 Amazonas {scene_id}",
        "dataProduct": sentinel_product_id,
        "featureCollection": {
            "type": "FeatureCollection",
            "features": [
                {
                    "type": "Feature",
                    "geometry": aoi["features"][0]["geometry"]
                }
            ]
        },
        "params": {
            "id": scene_id
        }
    }

    response = requests.post(url, json=payload, headers=headers_post)

    logger.info("Scene ID: %s -> Status code: %s (%s)", scene_id, response.status_code, response.text)

    if response.status_code not in [200, 202]:
        raise RuntimeError(
            f"Order creation failed for {scene_id}: {response.status_code} {response.text}"
        )

    data = response.json()

    order_id = data["results"][0]["id"]

    if not order_id:
        raise RuntimeError(f"Missing order id for {scene_id}: {data}")

    order_ids.append(order_id)

logger.info("All order IDs: %s", order_ids)

In [ ]:
# Check order status based on order ID

url_base = "https://api.up42.com/v2/orders"

for order_id in order_ids:
    url = f"{url_base}/{order_id}"
    response = requests.get(url, headers=headers_get)

    if response.status_code != 200:
        print("Failed to fetch status:", order_id, response.text)
        continue

    logger.info("ORDER: %s", order_id)
    logger.info("STATUS CODE: %s", response.status_code)

    data = response.json()

    status = (
        data.get("result", {}).get("status")
        or data.get("status")
    )

    logger.info("STATUS: %s", status)

In [ ]:
#######################################
## BLOCK 2: TO DOWNLOAD ORDERED DATA ##
#######################################

# STAC Collection
# Source: https://developer.up42.com/reference/getsascollections?utm_source=documentation

url = "https://api.up42.com/v2/assets/stac/collections"
params = {
    "limit": 10
}

response = requests.get(
    url,
    headers=headers_get,
    params=params
)

# Print status code
logger.info("Status code: %s", response.status_code)

collections = response.json()["collections"]

logger.info("Number of downloaded collections: %s", len(collections))

for c in collections:
    logger.info("ID: %s", c.get("id"))
    logger.info("Description: %s", c.get("description"))
    logger.info("Extent: %s", c.get("extent"))
    logger.info("-"* 50)

In [ ]:
# Explore STAC API metadata

# Pick one collection
demo_collection = collections[0]
demo_collection_id = demo_collection["id"]

# Explore the selected COLLECTION
logger.info("EXPLORE A COLLECTION")
logger.info(f"COLLECTION keys: {list(demo_collection.keys())}")
logger.info(f"Type: {demo_collection.get('type')}")
logger.info(f"Description: {demo_collection.get('description')}")
logger.info(f"Keywords: {demo_collection.get('keywords')}")
logger.info(f"Selected collection ID: {demo_collection_id}")
logger.info(f"="*50)

# Go one level deeper, explore FEATURES within the COLLECTION. In this case it is practically a SCENE with BANDS inside.
url = f"https://api.up42.com/v2/assets/stac/collections/{demo_collection_id}/items"
stac_response = requests.get(url, headers=headers_get)
data = stac_response.json()
logger.info("FEATURES (SCENES)")
logger.info(f"FEATURE keys: {list(data.keys())}")
logger.info(f"Type: {data.get('type')}")
scenes = data["features"]
logger.info(f"Number of features (scenes): {len(scenes)}")
logger.info(f"="*50)

# Explore a FEATURE (scene)
scene = scenes[0]
logger.info("EXPLORE A FEATURE")
logger.info(f"FEATURE keys: {list(scene.keys())}")
logger.info(f"ID: {scene['id']}")
logger.info(f"Datetime: {scene['properties'].get('datetime')}")
logger.info(f"Platform: {scene['properties'].get('platform')}")
logger.info(f"Cloud cover: {scene['properties'].get('eo:cloud_cover')}")
logger.info(f"Geometry type: {scene['geometry']['type']}")
logger.info(f"BBox: {scene['bbox']}")
logger.info(f"="*50)

# And one more level deeper: ASSETS
asset = scene["assets"]
logger.info("ASSETS (BANDS)")
logger.info(f"List of ASSETS: {list(asset.keys())}")
logger.info(f"="*50)

# Pick one ASSET
logger.info("EXPLORE AN ASSET")
b01 = asset.get("b01.tiff")
logger.info("b01.tiff asset details:")
logger.info(f"  Href: {b01.get('href')}")
logger.info(f"  Type: {b01.get('type')}")
logger.info(f"  Roles: {b01.get('roles')}")


In [ ]:
# Selecting the right collections

collection_id_list = []

for collection in collections:

# Structure template:
#    'extent': {
#        'spatial': {
#            'bbox': [[...]]},
#        'temporal': {
#            'interval': [['9999-99-99T99:99:99Z', '9999-99-99T99:99:99Z']]}},

    interval = collection.get("extent", {}).get("temporal", {}).get("interval", [])
    start_date = interval[0][0]
    month = start_date[5:7]

    if month == target_month:
        collection_id_list.append(collection["id"])

logger.info("Selected collection IDs: %s", collection_id_list)

In [ ]:
# Download STAC assets (Red & NIR bands) from each collection item
# TODO: cloud cover mask could be added to the downloaded band to improve processing output quality

for collection_id in collection_id_list:

    url = f"https://api.up42.com/v2/assets/stac/collections/{collection_id}/items"

    stac_response = requests.get(url, headers=headers_get)

    data = stac_response.json()
    scenes = data["features"]

    # Each timestamp has one scene with all the bands (assets) and other info
    for scene in scenes:
        assets = scene["assets"]
        timestamp = scene["properties"]["datetime"].split("T")[0]

        # Create URL and save each GeoTiff from the bands dictionary above
        for band_name, band_tiff_name in bands.items():

                if band_tiff_name not in assets:
                    raise ValueError(f"Missing band {band_tiff_name}")

                # Create a download URL
                # Source: https://developer.up42.com/reference/generatedownloadurlexternal?utm_source=documentation
                href = assets[band_tiff_name]["href"]
                download_url_endpoint = f"{href}/download-url"

                url_response = requests.post(download_url_endpoint, headers=headers_post)
                download_info = url_response.json()
        
                # Save file locally
                file_url = download_info["url"]
                file_response = requests.get(file_url)

                file_path = f"{raw_data_dir}/{collection_id}_{timestamp}_{band_name}.tif"
                with open(file_path, "wb") as f:
                    f.write(file_response.content)

                time.sleep(0.5)

In [ ]:
##############################
## BLOCK 3: TO PROCESS NDVI ##
##############################

# Scan downloaded files - define timestamps

red_files = glob.glob(os.path.join(raw_data_dir, "*_red.tif"))

scenes = []

for red_path in red_files:

    filename = os.path.basename(red_path)
    
    # Extract ID + timestamp
    # some-id_2025-07-06_red.tif >>> some-id
    parts = filename.split("_")
    collection_id = parts[0]
    # some-id_2025-07-06_red.tif >>> 2025-07-06
    timestamp = parts[1]
    
    nir_path = f"{raw_data_dir}/{collection_id}_{timestamp}_nir.tif"

    if os.path.exists(nir_path):
        scenes.append({
            "id": collection_id,
            "timestamp": timestamp,
            "red": red_path,
            "nir": nir_path
        })
    else:
        logger.error("Missing NIR for: %s", filename)

logger.info("Downloaded scenes: %s", scenes)

In [ ]:
# Calculate NDVI for each year

scene_stats_inputs = []

for scene in scenes:

    red_path = scene["red"]
    nir_path = scene["nir"]
    timestamp = scene["timestamp"]
    collection_id = scene["id"]

    output_path = f"{ndvi_data_dir}/{collection_id}_{timestamp}_ndvi.tif"
    
    if not os.path.exists(output_path):

        with rasterio.open(red_path) as red_dataset, rasterio.open(nir_path) as nir_dataset:

            assert red_dataset.shape == nir_dataset.shape
            assert red_dataset.transform == nir_dataset.transform

            profile = red_dataset.profile.copy()
            profile.update(dtype=rasterio.float32,count=1,compress='lzw')

            with rasterio.open(output_path, "w", **profile) as dst:

                # Iterate over blocks/windows
                for ji, window in red_dataset.block_windows(1):

                    red_band = red_dataset.read(1, window=window).astype(np.float32)
                    nir_band = nir_dataset.read(1, window=window).astype(np.float32)

                    mask = (red_band == 0) | (nir_band == 0)

                    ndvi = (nir_band - red_band) / (nir_band + red_band + 1e-6)
                    ndvi[mask] = np.nan

                    dst.write(ndvi, 1, window=window)

        logger.info("Finished processing NDVI for: %s - %s", collection_id, timestamp)

    else:
        logger.info("NDVI already calculated for: %s - %s", collection_id, timestamp)

    # Always append all relevant data for NDVI mean calculations in the next step
    scene_stats_inputs.append({
        "collection_id": collection_id,
        "timestamp": timestamp,
        "ndvi_path": output_path
    })

In [ ]:
# Calculate mean NDVI per scene

scene_stats = []

for scene in scene_stats_inputs:

    ndvi_path = scene["ndvi_path"]
    timestamp = datetime.strptime(scene["timestamp"], "%Y-%m-%d")

    with rasterio.open(ndvi_path) as src:
        ndvi = src.read(1).astype(np.float32)

        # Replace nodata (-9999) by NaN
        if src.nodata is not None:
            ndvi = np.where(ndvi == src.nodata, np.nan, ndvi)

        mean_ndvi = np.nanmean(ndvi)

    scene_stats.append({
        "collection_id": scene["collection_id"],
        "timestamp": timestamp,
        "mean_ndvi": mean_ndvi,
        "ndvi_path" : ndvi_path
    })

for c in scene_stats:
    logger.info("Collection ID: %s", c.get("collection_id"))
    logger.info("Timestamp: %s", c.get("timestamp"))
    logger.info("Mean NDVI: %s", c.get("mean_ndvi"))
    logger.info("NDVI path: %s", c.get("ndvi_path"))

In [ ]:
# Plot mean NDVI over time

# Sort NSVI scenes by time
timestamp = datetime.fromisoformat(parts[1])
scene_stats = sorted(scene_stats, key=lambda x: x["timestamp"])

logger.info("FIRST DATE: %s", scene_stats[0]["timestamp"])
logger.info("LAST DATE: %s", scene_stats[-1]["timestamp"])

dates = [s["timestamp"] for s in scene_stats]
means = [s["mean_ndvi"] for s in scene_stats]

plt.figure(figsize=(10, 5))

plt.plot(dates, means, marker="o", linewidth=2)

plt.title("Mean NDVI Over Time")
plt.xlabel("Date")
plt.ylabel("Mean NDVI")

plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.show()


In [ ]:
# Static NDVI maps (first year, last year)

# Sort NSVI scenes by time
scene_stats = sorted(scene_stats, key=lambda x: x["timestamp"])

logger.info("FIRST DATE: %s", scene_stats[0]["timestamp"])
logger.info("LAST DATE: %s", scene_stats[-1]["timestamp"])

first = scene_stats[0]
last = scene_stats[-1]

with rasterio.open(first["ndvi_path"]) as src:
    ndvi_first = src.read(1).astype(np.float32)
    if src.nodata is not None:
        ndvi_first = np.where(ndvi_first == src.nodata, np.nan, ndvi_first)

with rasterio.open(last["ndvi_path"]) as src:
    ndvi_last = src.read(1).astype(np.float32)
    if src.nodata is not None:
        ndvi_last = np.where(ndvi_last == src.nodata, np.nan, ndvi_last)

# Plot maps
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

vmin, vmax = -1, 1

im1 = axes[0].imshow(ndvi_first, cmap="RdYlGn", vmin=vmin, vmax=vmax)
axes[0].set_title(f"First NDVI\n{first['timestamp']}")
axes[0].axis("off")

im2 = axes[1].imshow(ndvi_last, cmap="RdYlGn", vmin=vmin, vmax=vmax)
axes[1].set_title(f"Last NDVI\n{last['timestamp']}")
axes[1].axis("off")

fig.colorbar(im1, ax=axes, fraction=0.03, pad=0.02, label="NDVI")

plt.show()


In [ ]:
# Calculate NDVI difference map between first and last year

# Ensure same shape
if ndvi_first.shape != ndvi_last.shape:
    raise ValueError(f"Shape mismatch: {ndvi_first.shape} vs {ndvi_last.shape}")

ndvi_diff = ndvi_last - ndvi_first
ndvi_diff = np.ma.masked_invalid(ndvi_diff)

fig, ax = plt.subplots(1, 1, figsize=(7, 6), constrained_layout=True)

# Diverging colormap centered at 0
cmap = plt.cm.BrBG

vmax = np.nanmax(np.abs(ndvi_diff))
vmin = -vmax

im = ax.imshow(ndvi_diff, cmap=cmap, vmin=vmin, vmax=vmax)

ax.set_title(
    f"NDVI Change (Last - First)\n"
    f"{last['timestamp']} − {first['timestamp']}"
)
ax.axis("off")

cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("NDVI Change")

plt.show()